# Super-resolution demo (scale x3)
Загружает обученную модель, прогоняет одну картинку: слева — LR (пикселями), справа — результат модели.

In [ ]:

import torch
from PIL import Image
import matplotlib.pyplot as plt
from pathlib import Path
from torchvision import transforms as T

from src.model import build_model
from src.utils import load_checkpoint

# Paths
ckpt_path = Path('checkpoints/sr_unet_x3.pt')
image_path = Path('data/sr/test_hr/100075.jpg')
scale = 3
device = 'cuda' if torch.cuda.is_available() else 'cpu'


class Cfg: ...
config = Cfg()
config.model = Cfg(); config.model.in_channels=3; config.model.out_channels=3; config.model.base_channels=64; config.model.bilinear=True
model = build_model(config).to(device).eval()
state = load_checkpoint(ckpt_path, map_location=device)
model.load_state_dict(state['model_state_dict'])

# Load image
hr = Image.open(image_path).convert('RGB')
# Make LR by downscaling
lr_small = hr.resize((hr.width // scale, hr.height // scale), Image.BICUBIC)
# Pixelated view for visualization (nearest upsample)
lr_pix = lr_small.resize(hr.size, Image.NEAREST)
# Model input: bicubic upsample back to HR size
lr_up = lr_small.resize(hr.size, Image.BICUBIC)

to_tensor = T.ToTensor()
lr_tensor = to_tensor(lr_up).unsqueeze(0).to(device)
with torch.no_grad():
    pred = model(lr_tensor).clamp(0, 1)

pred_img = T.ToPILImage()(pred[0].cpu())

# Plot
fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(lr_pix); axes[0].set_title('LR (pixelated)'); axes[0].axis('off')
axes[1].imshow(pred_img); axes[1].set_title('Model SR'); axes[1].axis('off')
plt.tight_layout()
plt.show()
